In [1]:
import pandas as pd
import numpy as np

In [2]:
gdp = pd.read_csv(
    "../Data/External/World Bank GDP Growth/API_NY.GDP.MKTP.KD.ZG_DS2_en_csv_v2_443095.csv",
    skiprows=4
)

In [3]:
gdp.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,3.493430,3.212471,1.225112,-23.897990,14.730616,10.636431,7.706798,6.810777,NaN,NaN
1,Africa Eastern and Southern,AFE,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,NaN,0.418937,7.937038,5.623764,4.649241,5.138168,...,2.677524,2.705194,2.030077,-2.817572,4.578772,3.722717,1.931160,2.763839,NaN,NaN
2,Afghanistan,AFG,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,2.647003,1.189228,3.911603,-2.351101,-20.738839,-6.240172,2.266944,NaN,NaN,NaN
3,Africa Western and Central,AFW,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,NaN,1.869593,3.726090,7.038388,5.364089,4.105339,...,2.296349,2.904664,3.281683,-3.730630,2.549691,4.472795,3.662428,4.585674,NaN,NaN
4,Angola,AGO,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.149396,-0.594411,-0.204680,-4.042447,2.102753,4.216003,1.263308,4.423907,NaN,NaN


In [4]:
gdp.shape

(266, 71)

In [5]:
gdp = gdp.drop(
    columns=[
        "Country Code",
        "Indicator Name",
        "Indicator Code",
        "Unnamed: 70"
    ]
)

In [6]:
gdp = gdp.melt(
    id_vars="Country Name",
    var_name="year",
    value_name="gdp_growth"
)

In [7]:
gdp["year"] = gdp["year"].astype(int)

In [9]:
retail = pd.read_csv("../Data/Processed/cleaned_superstore.csv")

In [10]:
retail.head()

,category,city,country,customer_id,customer_name,discount,market,order_date,order_id,order_priority,...,sales_bucket,profit_bucket,shipping_cost_bucket,average_order_value,shipping_cost_pct,discount_amount,net_sales,profit_status,delivery_status,profit_per_unit
0,Office Supplies,Los Angeles,United States,LS-172304,Lycoris Saunders,0.0,US,07-01-2011,CA-2011-130813,High,...,Low,High,Medium,6.333333,23.000000,0.0,19.0,Profit,Fast,3.1104
1,Office Supplies,Los Angeles,United States,MV-174854,Mark Van Huff,0.0,US,21-01-2011,CA-2011-148614,Medium,...,Low,High,Low,9.500000,4.947368,0.0,19.0,Profit,Normal,4.6464
2,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,05-08-2011,CA-2011-118962,Medium,...,Low,High,Low,7.000000,8.619048,0.0,21.0,Profit,Normal,3.2806
3,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,05-08-2011,CA-2011-118962,Medium,...,High,Very High,Medium,55.500000,4.135135,0.0,111.0,Profit,Normal,26.6304
4,Office Supplies,Los Angeles,United States,AP-109154,Arthur Prichep,0.0,US,29-09-2011,CA-2011-146969,High,...,Low,Medium,Low,6.000000,22.000000,0.0,6.0,Profit,Normal,3.1104


In [11]:
min_year = retail["year"].min()
max_year = retail["year"].max()

gdp = gdp[
    gdp["year"].between(min_year, max_year)
]

In [12]:
gdp = gdp.rename(
    columns={
        "Country Name": "country"
    }
)

In [14]:
gdp.head()

,country,year,gdp_growth
13566,Aruba,2011,3.369237
13567,Africa Eastern and Southern,2011,4.077477
13568,Afghanistan,2011,0.426355
13569,Africa Western and Central,2011,4.922047
13570,Angola,2011,3.593870


In [15]:
country_mapping = {
    "Hong Kong": "Hong Kong SAR, China",
    "Democratic Republic of the Congo": "Congo, Dem. Rep.",
    "Republic of the Congo": "Congo, Rep.",
    "Egypt": "Egypt, Arab Rep.",
    "Iran": "Iran, Islamic Rep.",
    "South Korea": "Korea, Rep.",
    "North Korea": "Korea, Dem. People's Rep.",
    "Russia": "Russian Federation",
    "Vietnam": "Viet Nam",
    "Venezuela": "Venezuela, RB",
    "Macedonia": "North Macedonia",
    "Czech Republic": "Czechia",
    "Turkey": "Türkiye",
    "Argentina": "Argentine Republic",
    "Myanmar (Burma)": "Myanmar",
    "Kyrgyzstan": "Kyrgyz Republic",
    "Slovakia": "Slovak Republic",
    "Swaziland": "Eswatini"
}

retail["country"] = retail["country"].replace(country_mapping)

In [16]:
retail_gdp = pd.merge(
    retail,
    gdp,
    on=["country", "year"],
    how="left"
)

In [17]:
retail_gdp["gdp_growth"].isnull().sum()

np.int64(1940)

In [18]:
retail_gdp[
    retail_gdp["gdp_growth"].isnull()
]["country"].value_counts()

country
Türkiye               1378
Argentine Republic     390
Somalia                 52
Yemen                   30
Syria                   29
Martinique              25
Taiwan                  14
Djibouti                12
Guadeloupe               8
Eritrea                  2
Name: count, dtype: int64

In [19]:
gdp[gdp["country"].str.contains("Turkey|Türkiye", case=False, na=False)]["country"].unique()

array([], dtype=object)

In [22]:
gdp[gdp["country"].str.contains("Tur", case=False, na=False)]["country"].unique()

array(['Turks and Caicos Islands', 'Turkmenistan', 'Turkiye'],
      dtype=object)

In [23]:
retail[retail["country"] == "Türkiye"].head()


,category,city,country,customer_id,customer_name,discount,market,order_date,order_id,order_priority,...,sales_bucket,profit_bucket,shipping_cost_bucket,average_order_value,shipping_cost_pct,discount_amount,net_sales,profit_status,delivery_status,profit_per_unit
29302,Furniture,Sivas,Türkiye,PA-90602,Pete Armstrong,0.6,EMEA,25-04-2011,TU-2011-7670,Medium,...,Medium,Low,Medium,23.00,6.673913,27.6,18.4,Loss,Normal,-21.978
29303,Technology,Ankara,Türkiye,LC-69602,Lindsay Castell,0.6,EMEA,26-04-2011,TU-2011-3910,High,...,Very High,Low,Very High,103.25,14.871671,247.8,165.2,Loss,Fast,-98.010
29304,Technology,Kartal,Türkiye,CG-20402,Catherine Glotzbach,0.6,EMEA,15-07-2011,TU-2011-9120,Critical,...,High,Low,Very High,70.50,30.418440,84.6,56.4,Loss,Fast,-59.922
29305,Furniture,Gebze,Türkiye,PR-88803,Patrick Ryan,0.6,EMEA,16-07-2011,TU-2011-490,Critical,...,Very High,Low,Very High,134.00,22.518657,160.8,107.2,Loss,Fast,-63.654
29306,Furniture,Istanbul,Türkiye,MY-82952,Muhammed Yedwab,0.6,EMEA,17-08-2011,TU-2011-660,High,...,Very High,Low,Very High,144.75,20.184801,347.4,231.6,Loss,Fast,-90.486


In [36]:
retail["country"] = retail["country"].replace({
    "Türkiye": "Turkiye"
})

In [37]:
retail_gdp = pd.merge(
    retail,
    gdp,
    on=["country", "year"],
    how="left"
)

In [38]:
retail_gdp["gdp_growth"].isnull().sum()

np.int64(562)

In [39]:
country_mapping.pop("Argentina", None)

retail["country"] = retail["country"].replace(country_mapping)

In [40]:
retail_gdp = pd.merge(
    retail,
    gdp,
    on=["country", "year"],
    how="left"
)

In [41]:
retail_gdp["gdp_growth"].isnull().sum()

np.int64(562)

In [43]:
retail_gdp[
    (retail_gdp["country"] == "Argentina") &
    (retail_gdp["gdp_growth"].isnull())
]

,category,city,country,customer_id,customer_name,discount,market,order_date,order_id,order_priority,...,profit_bucket,shipping_cost_bucket,average_order_value,shipping_cost_pct,discount_amount,net_sales,profit_status,delivery_status,profit_per_unit,gdp_growth


In [44]:
retail_gdp[
    (retail_gdp["country"] == "Argentina") &
    (retail_gdp["gdp_growth"].isnull())
].shape

(0, 48)

In [42]:
retail.head()

,category,city,country,customer_id,customer_name,discount,market,order_date,order_id,order_priority,...,sales_bucket,profit_bucket,shipping_cost_bucket,average_order_value,shipping_cost_pct,discount_amount,net_sales,profit_status,delivery_status,profit_per_unit
0,Office Supplies,Los Angeles,United States,LS-172304,Lycoris Saunders,0.0,US,07-01-2011,CA-2011-130813,High,...,Low,High,Medium,6.333333,23.000000,0.0,19.0,Profit,Fast,3.1104
1,Office Supplies,Los Angeles,United States,MV-174854,Mark Van Huff,0.0,US,21-01-2011,CA-2011-148614,Medium,...,Low,High,Low,9.500000,4.947368,0.0,19.0,Profit,Normal,4.6464
2,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,05-08-2011,CA-2011-118962,Medium,...,Low,High,Low,7.000000,8.619048,0.0,21.0,Profit,Normal,3.2806
3,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,05-08-2011,CA-2011-118962,Medium,...,High,Very High,Medium,55.500000,4.135135,0.0,111.0,Profit,Normal,26.6304
4,Office Supplies,Los Angeles,United States,AP-109154,Arthur Prichep,0.0,US,29-09-2011,CA-2011-146969,High,...,Low,Medium,Low,6.000000,22.000000,0.0,6.0,Profit,Normal,3.1104


In [32]:
gdp.to_csv(
    "../Data/Processed/gdp_growth_cleaned.csv",
    index=False
)